-- Exploratory Data Analysis (EDA) 
-- Bronze Layer Data Quality Audit
-- Target Table: bronze.raw_sales
-- Objective: Audit volumetrics, duplicates, schema integrity, and business rules

In [0]:
-- ------------------------------------------------------------------------------
-- CHECK 1: Volumetrics & Primary Key Completeness
-- COMPROBACIÓN 1: Volumetría e integridad de la clave primaria
-- ------------------------------------------------------------------------------
SELECT 
    COUNT(*) AS total_records,
    COUNT(order_id) AS non_null_order_ids,
    COUNT(DISTINCT order_id) AS unique_order_ids,
    COUNT(*) - COUNT(order_id) AS null_order_ids,
    COUNT(*) - COUNT(customer_id) AS null_customer_ids
FROM ecommerce_bronze.raw_sales_analytics;

In [0]:
-- ------------------------------------------------------------------------------
-- CHECK 2: Duplicate Primary Keys Audit
-- COMPROBACIÓN 2: Auditoría de claves primarias duplicadas
-- ------------------------------------------------------------------------------
SELECT 
    order_id, 
    COUNT(*) AS occurrence_count
FROM ecommerce_bronze.raw_sales_analytics
GROUP BY order_id
HAVING COUNT(*) > 1;

In [0]:
-- ------------------------------------------------------------------------------
-- CHECK 3: Null Values & Missing Data Scan Across All Columns
-- COMPROBACIÓN 3: Análisis de valores nulos y datos faltantes en todas las columnas
-- ------------------------------------------------------------------------------
SELECT 
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
    SUM(CASE WHEN order_date IS NULL THEN 1 ELSE 0 END) AS null_order_date,
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
    SUM(CASE WHEN product_category IS NULL THEN 1 ELSE 0 END) AS null_category,
    SUM(CASE WHEN unit_price IS NULL THEN 1 ELSE 0 END) AS null_price,
    SUM(CASE WHEN quantity IS NULL THEN 1 ELSE 0 END) AS null_quantity,
    SUM(CASE WHEN discount IS NULL THEN 1 ELSE 0 END) AS null_discount,
    SUM(CASE WHEN revenue IS NULL THEN 1 ELSE 0 END) AS null_revenue
FROM ecommerce_bronze.raw_sales_analytics;

In [0]:
-- ------------------------------------------------------------------------------
-- CHECK 4: Boundary & Numerical Outliers Check
-- VERIFICACIÓN 4: Comprobación de límites y valores atípicos numéricos
-- ------------------------------------------------------------------------------
SELECT 
    MIN(unit_price) AS min_unit_price,
    MAX(unit_price) AS max_unit_price,
    MIN(quantity) AS min_quantity,
    MAX(quantity) AS max_quantity,
    MIN(discount) AS min_discount,
    MAX(discount) AS max_discount,
    MIN(revenue) AS min_revenue,
    MAX(revenue) AS max_revenue
FROM ecommerce_bronze.raw_sales_analytics;

In [0]:
-- CHECK 4.1: Boundary & Numerical Outliers Check (With Explicit Casting)
-- COMPROBACIÓN 4.1: Verificación de valores atípicos numéricos y de límites (con conversión explícita de tipo)
-- ------------------------------------------------------------------------------
SELECT 
    MIN(CAST(unit_price AS DECIMAL(10,2))) AS min_unit_price,
    MAX(CAST(unit_price AS DECIMAL(10,2))) AS max_unit_price,
    MIN(CAST(quantity AS INT))             AS min_quantity,
    MAX(CAST(quantity AS INT))             AS max_quantity,
    MIN(CAST(discount AS DECIMAL(5,2)))    AS min_discount,
    MAX(CAST(discount AS DECIMAL(5,2)))    AS max_discount,
    MIN(CAST(revenue AS DECIMAL(10,2)))    AS min_revenue,
    MAX(CAST(revenue AS DECIMAL(10,2)))    AS max_revenue
FROM ecommerce_bronze.raw_sales_analytics;

In [0]:
-- CHECK 5: Financial Formula & Revenue Integrity (With Explicit Casting)
-- COMPROBACIÓN 5: Fórmula financiera e integridad de los ingresos (con conversión explícita de tipos)
-- ------------------------------------------------------------------------------
SELECT 
    order_id,
    CAST(revenue AS DECIMAL(10,2)) AS revenue,
    ROUND(CAST(quantity AS INT) * CAST(unit_price AS DECIMAL(10,2)) * (1 - CAST(discount AS DECIMAL(5,2))), 2) AS calculated_revenue,
    ABS(CAST(revenue AS DECIMAL(10,2)) - ROUND(CAST(quantity AS INT) * CAST(unit_price AS DECIMAL(10,2)) * (1 - CAST(discount AS DECIMAL(5,2))), 2)) AS discrepancy
FROM ecommerce_bronze.raw_sales_analytics
WHERE ABS( CAST(revenue AS DECIMAL(10,2)) - ROUND(CAST(quantity AS INT) * CAST(unit_price AS DECIMAL(10,2)) * (1 - CAST(discount AS DECIMAL(5,2))), 2)) > 0.01;